In [22]:
import pprint
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

import os

from dotenv import load_dotenv

load_dotenv("C:\\Users\\socgen\\ML\\agentic_ai_and_ops\\langchain_day5\\.env")

True

In [23]:
model_gr_lamma = init_chat_model("llama-3.3-70b-versatile",
                        api_key=os.environ["GROQ_API_KEY"],
                        model_provider="groq",
                        # base_url="https://api.groq.com/openai/v1",
                        max_tokens=1000, temperature=0.0)

model_or_paid_gpt_luna_pro = init_chat_model("openai/gpt-5.6-luna-pro",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=100, temperature=0.0)

model_or_free_nvidia = init_chat_model("nvidia/nemotron-3-ultra-550b-a55b:free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)


model_or_free = init_chat_model("openrouter/free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)


model_ollama = init_chat_model("ollama:gemma4:latest",
                                max_tokens=200, 
                                temperature=0.0)

In [25]:
from pydantic import BaseModel, Field
from typing import Literal

from langchain.agents import create_agent

from langchain.agents.structured_output import ToolStrategy

In [40]:
class ProductReview(BaseModel):
    """Analysis of a product review."""
    product_name: str = Field(description="The name of the product")
    review_text: str = Field(description="The text of the review")
    rating: Literal[1, 2, 3, 4, 5] = Field(description="The rating given by the reviewer")
    shipping_sentiment: Literal["positive", "negative", "neutral"] = Field(description="The sentiment of the review regarding shipping")
    pricing_sentiment: Literal["positive", "negative", "neutral"] = Field(description="The sentiment of the review regarding pricing")
    quality_sentiment: Literal["positive", "negative", "neutral"] = Field(description="The sentiment of the review regarding quality")

In [41]:
agent = create_agent(
    model=model_gr_lamma,
    tools=[],
    response_format=ToolStrategy(ProductReview)
)

In [42]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "Analyze this review: 'Great Mobile: 5 out of 5 stars. Fast shipping, but expensive'"}]
})

In [43]:
from pprint import pprint

pprint(result, depth=4, compact=True)

{'messages': [HumanMessage(content="Analyze this review: 'Great Mobile: 5 out of 5 stars. Fast shipping, but expensive'", additional_kwargs={}, response_metadata={}, id='28fc93f9-ecc5-4a43-8247-ed2d8bb7357c'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'v12gwm4dv', 'function': {'arguments': '{"pricing_sentiment":"negative","product_name":"Mobile","quality_sentiment":"positive","rating":5,"review_text":"Great Mobile: 5 out of 5 stars. Fast shipping, but expensive","shipping_sentiment":"positive"}', 'name': 'ProductReview'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 396, 'total_tokens': 461, 'completion_time': 0.148917015, 'completion_tokens_details': None, 'prompt_time': 0.043384672, 'prompt_tokens_details': None, 'queue_time': 0.058820356, 'total_time': 0.192301687}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'too

In [46]:
result["structured_response"]

ProductReview(product_name='Mobile', review_text='Great Mobile: 5 out of 5 stars. Fast shipping, but expensive', rating=5, shipping_sentiment='positive', pricing_sentiment='negative', quality_sentiment='positive')

In [52]:
result["structured_response"].model_dump() # Convert the result to a dictionary

{'product_name': 'Mobile',
 'review_text': 'Great Mobile: 5 out of 5 stars. Fast shipping, but expensive',
 'rating': 5,
 'shipping_sentiment': 'positive',
 'pricing_sentiment': 'negative',
 'quality_sentiment': 'positive'}

In [53]:
result["structured_response"].model_dump_json()

'{"product_name":"Mobile","review_text":"Great Mobile: 5 out of 5 stars. Fast shipping, but expensive","rating":5,"shipping_sentiment":"positive","pricing_sentiment":"negative","quality_sentiment":"positive"}'